### 0 - Dependëncias

In [ ]:
# Dependencias

import pandas as pd
import os
import unicodedata
import re

### 1 - Carregar dados coletados do site ANATEL - todos os estados, 8 indicadores, 2022

In [74]:
# Lista dos arquivos (ajuste os nomes conforme seu diretório)
arquivos = [
    'RQUAL_8ind-A.xlsx',
    'RQUAL_8ind-SP.xlsx',
    'RQUAL_8ind-RN-RO-RR.xlsx',
    'RQUAL_8ind-E-G-MA.xlsx',
    'RQUAL_8ind-MT-MS.xlsx',
    'RQUAL_8ind-BA.xlsx',
    'RQUAL_8ind-P.xlsx',
    'RQUAL_8ind-SC-SE-TO.xlsx',
    'RQUAL_8ind-CD.xlsx',
    'RQUAL_8ind-RJ.xlsx',
    'RQUAL_8ind-MG.xlsx',
    'RQUAL_8ind-RS.xlsx'
]

# Carregar todos os DataFrames
dfs = []
for arquivo in arquivos:
    try:
        df = pd.read_excel(arquivo)
        df['Fonte'] = arquivo  # Adicionar coluna com o nome do arquivo de origem
        dfs.append(df)
    except Exception as e:
        print(f"Erro ao carregar {arquivo}: {e}")

KeyboardInterrupt: 

In [ ]:
# Concatenar todos os DataFrames
df_consolidado = pd.concat(dfs, ignore_index=True)

# Verificar resultado
print(f"Total de linhas: {len(df_consolidado)}")
df_consolidado.head()

Total de linhas: 5962723


,Ano,Mês,Serviço,Tipo,Grupo,UF,Município,Código IBGE,Prestadora,Indicador,Descrição,Resultado,Unidade de Medida,Valor de Referência Superior,Valor de Referência Inferior,Erro Amostral,Limite Inferior,Limite Superior,Fonte
0,2022,3,Banda Larga Fixa,Indicador IQS,Redes,AL,Maceió,2704302,ALGAR,IND5,Latência bidirecional da conexão de dados,0,%,0.95,0.65,70.7,0,70.7083,RQUAL_8ind-A.xlsx
1,2022,3,Banda Larga Fixa,Indicador IQS,Redes,AP,Macapá,1600303,CLARO,IND5,Latência bidirecional da conexão de dados,100,%,0.95,0.65,31.62,68.3783,100,RQUAL_8ind-A.xlsx
2,2022,3,Banda Larga Fixa,Indicador IQS,Redes,AL,Maceió,2704302,CLARO,IND5,Latência bidirecional da conexão de dados,78.7878,%,0.95,0.65,10.65,68.1282,89.4474,RQUAL_8ind-A.xlsx
3,2022,3,Banda Larga Fixa,Indicador IQS,Redes,AM,Manaus,1302603,CLARO,IND5,Latência bidirecional da conexão de dados,68.7774,%,0.95,0.65,3.63,65.1406,72.4142,RQUAL_8ind-A.xlsx
4,2022,3,Banda Larga Fixa,Indicador IQS,Redes,AC,Rio Branco,1200401,CLARO,IND5,Latência bidirecional da conexão de dados,76.1904,%,0.95,0.65,15.42,60.7606,91.6202,RQUAL_8ind-A.xlsx


In [ ]:
# Exportar para CSV
df_consolidado.to_csv(
    "Tabela_CSV_Indicadores_RQUAL_consolidados20250502.csv",  # Nome do arquivo
    index=False,                          # Não incluir índice
    encoding='utf-8',                     # Codificação para caracteres especiais (acentos)
    sep=';'                               # Separador compatível com Excel BR
)

# Verificar se o arquivo foi criado
import os
print(f"Arquivo gerado: {os.path.exists('Tabela_CSV_Indicadores_RQUAL_consolidados20250502.csv')}")
print(f"Tamanho: {os.path.getsize('Tabela_CSV_Indicadores_RQUAL_consolidados20250502.csv') / 1024:.2f} KB")

Arquivo gerado: True
Tamanho: 926786.61 KB


### 2 - Avaliação da consistencia da base RQUAL de 2022

In [24]:
# print(df_ibge.columns)
# print(df_rqual.columns)

In [25]:
# --- Constantes ---
# Nomes de arquivos (melhor definir como constantes para fácil alteração)
ARQUIVO_IBGE = 'base_integrada_ibge_municipios.csv'
ARQUIVO_RQUAL = 'Tabela_CSV_Indicadores_RQUAL_consolidados20250502.csv'

# Nomes de colunas relevantes para clareza e fácil refatoração
# Colunas em df_rqual
COL_RQUAL_COD_IBGE = 'Código IBGE'
COL_RQUAL_MUNICIPIO = 'Município'
COL_RQUAL_UF = 'UF'
COL_RQUAL_NOME_MUN_NORM = 'nome_mun_norm' # Coluna normalizada que será criada/usada

# Colunas em df_ibge
COL_IBGE_COD_MUN = 'cod_mun'           # Equivalente ao 'Código IBGE' do RQUAL
COL_IBGE_NOME_MUN = 'nome_mun'         # Fonte para 'Município'
COL_IBGE_SIGLA_UF = 'sigla_uf'         # Fonte para 'UF' (geralmente preferível a cod_uf para exibição)
# COL_IBGE_COD_UF = 'cod_uf'           # Alternativa para UF, se sigla_uf não estiver disponível
COL_IBGE_NOME_MUN_NORM = 'nome_mun_norm' # Coluna normalizada que será criada/usada

In [28]:
# --- Funções Auxiliares ---

def normalizar_texto(texto):
    """
    Normaliza uma string: remove acentos, converte para minúsculas,
    remove caracteres especiais (exceto alphanumeric e espaço) e espaços extras.
    Retorna uma string vazia se a entrada for NaN.
    """
    if pd.isna(texto):
        return ""  # Retorna string vazia para NaN para evitar erros
    
    texto_str = str(texto) # Garante que o input é uma string
    texto_str = texto_str.strip().lower()
    # Remove acentos
    texto_str = unicodedata.normalize('NFKD', texto_str)
    texto_str = ''.join(c for c in texto_str if not unicodedata.combining(c))
    # Remove caracteres não alfanuméricos (exceto espaço)
    texto_str = re.sub(r'[^a-z0-9 ]', '', texto_str)
    # Substitui múltiplos espaços por um único espaço
    texto_str = re.sub(r'\s+', ' ', texto_str)
    return texto_str.strip()

def carregar_e_preparar_dados(caminho_ibge, caminho_rqual):
    """
    Carrega os dataframes IBGE e RQUAL e realiza a preparação inicial
    das colunas chave para o merge e preenchimento.
    """
    print(f"Carregando dados do IBGE de: {caminho_ibge}")
    df_ibge = pd.read_csv(caminho_ibge)
    print(f"Carregando dados do RQUAL de: {caminho_rqual}")
    df_rqual = pd.read_csv(caminho_rqual, sep=';', encoding='utf-8')

    # --- Preparação df_ibge ---
    # Garantir que a coluna de código do município no IBGE seja numérica (idealmente int)
    # e tratar possíveis erros de conversão (ex: valores não numéricos)
    df_ibge[COL_IBGE_COD_MUN] = pd.to_numeric(df_ibge[COL_IBGE_COD_MUN], errors='coerce')
    df_ibge.dropna(subset=[COL_IBGE_COD_MUN], inplace=True) # Remove linhas onde cod_mun não pôde ser convertido
    df_ibge[COL_IBGE_COD_MUN] = df_ibge[COL_IBGE_COD_MUN].astype(int)

    # Normalizar nome do município e padronizar UF no df_ibge
    df_ibge[COL_IBGE_NOME_MUN_NORM] = df_ibge[COL_IBGE_NOME_MUN].apply(normalizar_texto)
    df_ibge[COL_IBGE_SIGLA_UF] = df_ibge[COL_IBGE_SIGLA_UF].astype(str).str.upper().str.strip()

    # --- Preparação df_rqual ---
    # Converter 'Código IBGE' do RQUAL para numérico.
    # 'coerce' transforma em NaN o que não puder ser convertido.
    df_rqual[COL_RQUAL_COD_IBGE] = pd.to_numeric(df_rqual[COL_RQUAL_COD_IBGE], errors='coerce')
    # Não removeremos NaNs aqui ainda, pois queremos tentar preenchê-los.

    # Normalizar nome do município e padronizar UF no df_rqual
    # É importante fazer isso ANTES de tentar preencher, para ter uma base consistente
    df_rqual[COL_RQUAL_NOME_MUN_NORM] = df_rqual[COL_RQUAL_MUNICIPIO].apply(normalizar_texto)
    
    # Padronizar UF, tratando strings 'nan' ou vazias como ausentes (pd.NA)
    df_rqual[COL_RQUAL_UF] = df_rqual[COL_RQUAL_UF].astype(str).str.upper().str.strip()
    df_rqual[COL_RQUAL_UF].replace(['NAN', ''], pd.NA, inplace=True)


    print("Dados carregados e colunas chave preparadas.")
    return df_ibge, df_rqual

In [30]:
# --- Lógica Principal ---

def preencher_dados_faltantes_rqual(df_rqual, df_ibge):
    """
    Preenche UF e Município ausentes no df_rqual utilizando dados do df_ibge,
    baseado no Código IBGE.
    """
    print("\n--- Iniciando preenchimento de dados faltantes no RQUAL ---")

    # 1. Diagnóstico Inicial: Contar valores ausentes em UF e Município no df_rqual
    # Consideramos ausente se for pd.NA (após padronização)
    ausentes_uf_antes = df_rqual[COL_RQUAL_UF].isna().sum()
    # Para município, consideramos ausente se o campo original for NaN ou a string normalizada for vazia
    ausentes_mun_antes = df_rqual[COL_RQUAL_MUNICIPIO].isna().sum()
    # ou (df_rqual[COL_RQUAL_NOME_MUN_NORM] == "") # Se normalizar_texto retorna "" para NaN

    print(f"Diagnóstico Inicial (df_rqual):")
    print(f"  Registros com UF ausente: {ausentes_uf_antes}")
    print(f"  Registros com Município ausente: {ausentes_mun_antes}")
    if df_rqual[COL_RQUAL_COD_IBGE].isna().any():
        print(f"  Atenção: {df_rqual[COL_RQUAL_COD_IBGE].isna().sum()} registros em RQUAL não possuem Código IBGE e não poderão ser preenchidos por este método.")

    # 2. Preparar df_ibge para consulta (lookup table)
    # Selecionar colunas relevantes e remover duplicatas baseadas no código do município do IBGE.
    # Isso garante que cada código de município tenha uma única entrada de nome e UF.
    df_ibge_lookup = df_ibge[[COL_IBGE_COD_MUN, COL_IBGE_NOME_MUN, COL_IBGE_SIGLA_UF]].copy()
    df_ibge_lookup.drop_duplicates(subset=[COL_IBGE_COD_MUN], keep='first', inplace=True)
    
    # Renomear colunas do df_ibge_lookup para o merge, para clareza e para evitar conflitos
    # se df_rqual já tivesse colunas com esses nomes (embora não seja o caso aqui para _ibge_fill)
    df_ibge_lookup = df_ibge_lookup.rename(columns={
        COL_IBGE_COD_MUN: COL_RQUAL_COD_IBGE, # Chave de merge
        COL_IBGE_NOME_MUN: 'nome_mun_ibge_fill',
        COL_IBGE_SIGLA_UF: 'uf_ibge_fill'
    })

    # 3. Realizar o merge para trazer os dados de referência do IBGE para o RQUAL
    # Usamos 'left' merge para manter todos os registros do df_rqual.
    # As colunas 'nome_mun_ibge_fill' e 'uf_ibge_fill' serão adicionadas ao df_rqual.
    # Elas terão valores onde houver correspondência de COL_RQUAL_COD_IBGE, e NaN onde não houver.
    df_rqual_merged = pd.merge(
        df_rqual,
        df_ibge_lookup,
        on=COL_RQUAL_COD_IBGE,
        how='left'
    )

    # 4. Preencher os valores ausentes
    
    # Preencher UF
    # Identificar linhas onde UF está ausente no RQUAL e existe um valor de preenchimento do IBGE
    idx_uf_a_preencher = df_rqual_merged[df_rqual_merged[COL_RQUAL_UF].isna() & \
                                         df_rqual_merged['uf_ibge_fill'].notna()].index
    if not idx_uf_a_preencher.empty:
        df_rqual_merged.loc[idx_uf_a_preencher, COL_RQUAL_UF] = df_rqual_merged.loc[idx_uf_a_preencher, 'uf_ibge_fill']
    
    # Preencher Município
    # Identificar linhas onde Município está ausente no RQUAL e existe um valor de preenchimento do IBGE
    idx_mun_a_preencher = df_rqual_merged[df_rqual_merged[COL_RQUAL_MUNICIPIO].isna() & \
                                          df_rqual_merged['nome_mun_ibge_fill'].notna()].index
    if not idx_mun_a_preencher.empty:
        df_rqual_merged.loc[idx_mun_a_preencher, COL_RQUAL_MUNICIPIO] = df_rqual_merged.loc[idx_mun_a_preencher, 'nome_mun_ibge_fill']
        # Se o município foi preenchido, precisamos re-normalizar seu nome
        df_rqual_merged.loc[idx_mun_a_preencher, COL_RQUAL_NOME_MUN_NORM] = \
            df_rqual_merged.loc[idx_mun_a_preencher, COL_RQUAL_MUNICIPIO].apply(normalizar_texto)

    # 5. Relatório de preenchimento e limpeza
    ausentes_uf_depois = df_rqual_merged[COL_RQUAL_UF].isna().sum()
    ausentes_mun_depois = df_rqual_merged[COL_RQUAL_MUNICIPIO].isna().sum()
    
    preenchidos_uf = ausentes_uf_antes - ausentes_uf_depois
    preenchidos_mun = ausentes_mun_antes - ausentes_mun_depois

    print("\nDiagnóstico Pós-Preenchimento (df_rqual_merged):")
    print(f"  Registros com UF ausente: {ausentes_uf_depois} (preenchidos: {preenchidos_uf})")
    print(f"  Registros com Município ausente: {ausentes_mun_depois} (preenchidos: {preenchidos_mun})")

    # Remover colunas auxiliares do merge (as que vieram do IBGE para preenchimento)
    df_rqual_final = df_rqual_merged.drop(columns=['nome_mun_ibge_fill', 'uf_ibge_fill'])
    
    print("Preenchimento concluído.")
    return df_rqual_final

In [42]:
# --- Bloco de Execução Principal ---
if __name__ == "__main__":
    # 1. Carregar e preparar dados
    df_ibge, df_rqual_original = carregar_e_preparar_dados(ARQUIVO_IBGE, ARQUIVO_RQUAL)

    # (Opcional) Fazer uma cópia para preservar o df_rqual original se precisar comparar depois
    # df_rqual_para_processar = df_rqual_original.copy()

    # 2. Preencher dados faltantes
    df_rqual_corrigido = preencher_dados_faltantes_rqual(df_rqual_original, df_ibge)

    # 3. (Opcional) Verificar alguns exemplos
    # print("\n--- Exemplo de dados em df_rqual_corrigido ---")
    # print(df_rqual_corrigido[[COL_RQUAL_COD_IBGE, COL_RQUAL_MUNICIPIO, COL_RQUAL_UF, COL_RQUAL_NOME_MUN_NORM]].head())


    
    print("\n--- Análise de consistência adicional (Opcional) ---")
    
    # 3.1 Identificar registros com UF ainda ausente no RQUAL_corrigido (após tentativa de preenchimento)
    uf_ainda_ausente_rqual = df_rqual_corrigido[df_rqual_corrigido[COL_RQUAL_UF].isna()]
    print(f"\n⚠️ Registros com UF AINDA ausente no RQUAL corrigido: {len(uf_ainda_ausente_rqual)}")
    if len(uf_ainda_ausente_rqual) > 0:
        print("📌 Exibindo os primeiros registros com UF ainda ausente (pode ser por falta de Código IBGE ou Código IBGE não encontrado na base IBGE):")
        print(uf_ainda_ausente_rqual[[COL_RQUAL_COD_IBGE, COL_RQUAL_MUNICIPIO, 'Ano', 'Mês', 'Indicador']].head())

Carregando dados do IBGE de: base_integrada_ibge_municipios.csv
Carregando dados do RQUAL de: Tabela_CSV_Indicadores_RQUAL_consolidados20250502.csv


/var/folders/f6/v3nynmvx48z8z0g2dt5fyw300000gn/T/ipykernel_11187/168581163.py:56: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_rqual[COL_RQUAL_UF].replace(['NAN', ''], pd.NA, inplace=True)


Dados carregados e colunas chave preparadas.

--- Iniciando preenchimento de dados faltantes no RQUAL ---
Diagnóstico Inicial (df_rqual):
  Registros com UF ausente: 0
  Registros com Município ausente: 0

Diagnóstico Pós-Preenchimento (df_rqual_merged):
  Registros com UF ausente: 0 (preenchidos: 0)
  Registros com Município ausente: 0 (preenchidos: 0)
Preenchimento concluído.

--- Análise de consistência adicional (Opcional) ---

⚠️ Registros com UF AINDA ausente no RQUAL corrigido: 0


## 3 - Salvar CSV e parquet

In [44]:
# 1. Salvar o dataframe corrigido
NOME_ARQUIVO_SAIDA_RQUAL = 'Tabela_CSV_Indicadores_RQUAL_consolidados20250516.csv'
df_rqual_corrigido.to_csv(NOME_ARQUIVO_SAIDA_RQUAL, sep=';', index=False, encoding='utf-8')
print(f"\nArquivo RQUAL corrigido salvo em: {NOME_ARQUIVO_SAIDA_RQUAL}")


Arquivo RQUAL corrigido salvo em: Tabela_CSV_Indicadores_RQUAL_consolidados20250516.csv


In [45]:

print(df_rqual_corrigido.columns)

print(df_rqual_corrigido.shape)

Index(['Ano', 'Mês', 'Serviço', 'Tipo', 'Grupo', 'UF', 'Município',
       'Código IBGE', 'Prestadora', 'Indicador', 'Descrição', 'Resultado',
       'Unidade de Medida', 'Valor de Referência Superior',
       'Valor de Referência Inferior', 'Erro Amostral', 'Limite Inferior',
       'Limite Superior', 'Fonte', 'nome_mun_norm'],
      dtype='object')
(5962723, 20)
